# Quantile regression

Quantile regression estimates a chosen quantile of `y` given the features `x`. This recipe shows the linear approach River already supports, where fitted quantiles are model-specific and conditional on `x`, plus the tree/leaf idea that follows the same conditional logic.

In [1]:
from __future__ import annotations

import itertools
import math
import random

from river import conf, linear_model, optim, preprocessing

## Linear quantile regression

In [2]:
model = linear_model.LinearRegression(loss=optim.losses.Quantile(alpha=0.9))

Here is a small stream where the conditional quantiles are known. The target is `2 * x + noise`, where `noise` follows an exponential distribution. Therefore the `alpha` quantile is `2 * x - log(1 - alpha)`.

In [3]:
def iter_data(seed=42):
    rng = random.Random(seed)
    while True:
        x = rng.uniform(-1, 1)
        yield {"x": x}, 2 * x + rng.expovariate(1)


def make_model(alpha):
    return preprocessing.StandardScaler() | linear_model.LinearRegression(
        optimizer=optim.SGD(0.005),
        intercept_lr=0.005,
        loss=optim.losses.Quantile(alpha=alpha),
    )

In [4]:
models = {alpha: make_model(alpha) for alpha in (0.1, 0.5, 0.9)}

for x, y in itertools.islice(iter_data(), 60_000):
    for model in models.values():
        model.learn_one(x, y)

xs = (-1, 0, 1)
print("alpha  estimated at x=-1,0,1        expected")
for alpha, model in models.items():
    estimated = [model.predict_one({"x": x}) for x in xs]
    expected = [2 * x - math.log1p(-alpha) for x in xs]
    estimated_text = "  ".join(f"{y:6.3f}" for y in estimated)
    expected_text = "  ".join(f"{y:6.3f}" for y in expected)
    print(f"{alpha:>4.1f}   {estimated_text}       {expected_text}")

alpha  estimated at x=-1,0,1        expected
 0.1   -1.891   0.119   2.128       -1.895   0.105   2.105
 0.5   -1.342   0.684   2.709       -1.307   0.693   2.693
 0.9    0.387   2.298   4.210        0.303   2.303   4.303


These fitted quantiles describe `Q(Y | X=x)`. They are not, by themselves, calibrated prediction intervals: the models are trained separately, quantile curves can cross, and there is no finite-sample coverage guarantee.

## Tree/leaf quantiles

A tree partitions the feature space into leaves. Instead of each leaf tracking only a mean prediction, a quantile-aware leaf can track the distribution of target values observed in that region. When a new `x` reaches a leaf, quantiles can be estimated from that leaf's target distribution.

This is feature-conditional because different leaves correspond to different regions of the feature space. It is also model-specific: the idea fits tree-based models naturally, but it is not a generic recipe for arbitrary estimators.

## RegressionJackknife

For estimator-agnostic intervals, River provides `conf.RegressionJackknife`. It wraps any regressor and tracks residual quantiles online.

In [5]:
base_model = preprocessing.StandardScaler() | linear_model.LinearRegression()
interval_model = conf.RegressionJackknife(regressor=base_model)

interval_model.learn_one({"x": 0}, 1.0)
interval_model.learn_one({"x": 1}, 3.0)

interval = interval_model.predict_one({"x": 0.5}, with_interval=True)
print(f"lower: {interval.lower:.3f}")
print(f"upper: {interval.upper:.3f}")

lower: 1.080
upper: 3.060


Those intervals are marginal rather than feature-conditional, so their width is not adjusted to each region of the feature space. A generic feature-conditional interval or quantile method remains a separate problem.